In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import classification_report
print("OK")

OK


In [2]:
import sys
sys.path.append("..")

from src.features import extract_symptoms
import pandas as pd

# load data
df_raw = pd.read_excel("../data/raw/covid.xlsx")

# run function
df_symptoms = extract_symptoms(df_raw)

# check output
df_symptoms.head()

,Fever,Myalgia,Chills,Fatigue,Sweating,Anorexia,Sore_Throat,Sneezing,Cough,Dyspnea,...,Ageusia,Chest_Pain,Palpitations,Nausea,Vomiting,Abdominal_Pain,Diarrhea,Depression,Sleep_Disorder,Headache
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,1,0,0,0,0,0,0,0,0
4,0,0,1,0,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
import sys
sys.path.append("..")

#import importlib
import src.features as features

#importlib.reload(features)

df_pmh = features.extract_pmh(df_raw)

print(df_pmh.shape)
df_pmh.head()

(16780, 28)


,Cancer,Organ_Transplant,Chronic_Kidney_Disease,Dialysis,Cancer_Type,Solid_Organ_Transplant,Asthma,Chronic_Respiratory_Disease,Hypertension,Myocardial_Infarction,...,Seizure,Diabetes_Type1,Diabetes_Type2,Hypothyroidism,Hyperthyroidism,Insulin_Use,Rheumatoid_Arthritis,Lupus,Behcet,Sjogren
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
#import importlib
import src.features as features

#importlib.reload(features)

df_vax = features.extract_vaccination(df_raw)

print(df_vax.shape)
df_vax.head()

(16780, 2)


,Vaccinated,Vaccine_Doses
0,0,0.0
1,0,0.0
2,0,0.0
3,0,0.0
4,1,2.0


In [5]:
print("Vaccinated %:", round(df_vax['Vaccinated'].mean() * 100, 2))
df_vax['Vaccine_Doses'].value_counts().sort_index()

Vaccinated %: 3.43


Vaccine_Doses
0.0    16205
1.0      107
2.0      354
3.0      106
4.0        8
Name: count, dtype: int64

In [6]:
import importlib
import src.features as features

importlib.reload(features)

df_final = features.build_features(df_raw)

print(df_final.shape)
df_final.head()

(16780, 55)


,Fever,Myalgia,Chills,Fatigue,Sweating,Anorexia,Sore_Throat,Sneezing,Cough,Dyspnea,...,Diabetes_Type2,Hypothyroidism,Hyperthyroidism,Insulin_Use,Rheumatoid_Arthritis,Lupus,Behcet,Sjogren,Vaccinated,Vaccine_Doses
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.0
2,1,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0.0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0.0
4,0,0,1,0,0,1,1,0,1,1,...,0,0,0,0,0,0,0,0,1,2.0


In [7]:
# 🎯 Step 1: Create target (CT lung involvement → severity)

df_target = df_raw['Unnamed: 69'].iloc[3:].reset_index(drop=True)

# Convert to binary:
# if there is any text → 1 (lung involvement)
# if empty → 0 (no involvement)

df_target = df_target.notna().astype(int)

# Check distribution
print("Target distribution:")
print(df_target.value_counts())

Target distribution:
Unnamed: 69
0    16588
1      192
Name: count, dtype: int64


In [13]:
#  feature و target

X = df_final.copy()
y = df_target

X = X.loc[y.index]

print(X.shape, y.shape)

(16780, 55) (16780,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [15]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("After SMOTE:\n", y_train_res.value_counts())

Before SMOTE:
 Unnamed: 69
0    13270
1      154
Name: count, dtype: int64
After SMOTE:
 Unnamed: 69
0    13270
1    13270
Name: count, dtype: int64


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=1000)

model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.69      0.82      3318
           1       0.02      0.42      0.03        38

    accuracy                           0.69      3356
   macro avg       0.50      0.56      0.42      3356
weighted avg       0.98      0.69      0.81      3356



In [17]:
from sklearn.metrics import classification_report

# Get predicted probabilities for the severe class (class 1)
y_proba = model.predict_proba(X_test)[:, 1]

# Try a custom threshold instead of the default 0.5
threshold = 0.2

# Convert probabilities to class labels based on the selected threshold
y_pred_threshold = (y_proba >= threshold).astype(int)

# Evaluate model performance with the new threshold
print(classification_report(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0       0.99      0.51      0.68      3318
           1       0.01      0.61      0.03        38

    accuracy                           0.51      3356
   macro avg       0.50      0.56      0.35      3356
weighted avg       0.98      0.51      0.67      3356



In [18]:
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred_threshold = (y_proba >= threshold).astype(int)
    
    print("Threshold:", threshold)
    print(classification_report(y_test, y_pred_threshold))
    print("-" * 50)

Threshold: 0.1
              precision    recall  f1-score   support

           0       0.99      0.42      0.59      3318
           1       0.01      0.74      0.03        38

    accuracy                           0.42      3356
   macro avg       0.50      0.58      0.31      3356
weighted avg       0.98      0.42      0.58      3356

--------------------------------------------------
Threshold: 0.2
              precision    recall  f1-score   support

           0       0.99      0.51      0.68      3318
           1       0.01      0.61      0.03        38

    accuracy                           0.51      3356
   macro avg       0.50      0.56      0.35      3356
weighted avg       0.98      0.51      0.67      3356

--------------------------------------------------
Threshold: 0.3
              precision    recall  f1-score   support

           0       0.99      0.58      0.73      3318
           1       0.01      0.55      0.03        38

    accuracy                       

In [20]:
!pip install xgboost

  Obtaining dependency information for xgboost from https://files.pythonhosted.org/packages/1f/3d/1661dd114a914a67e3f7ab66fa1382e7599c2a8c340f314ad30a3e2b4d08/xgboost-3.2.0-py3-none-win_amd64.whl.metadata
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 1.3 MB/s eta 0:01:20
   ---------------------------------------- 0.1/101.7 MB 1.1 MB/s eta 0:01:37
   ---------------------------------------- 0.2/101.7 MB 1.1 MB/s eta 0:01:29
   ---------------------------------------- 0.3/101.7 MB 1.3 MB/s eta 0:01:18
   ---------------------------------------- 0.4/101.7 MB 1.8 MB/s eta 0:00:58
   ---------------------------------------- 0.7/101.7 MB 2.3 MB/s eta 0:00:44
   ---------------------------------------- 0.7/101.7 MB 2.3 MB/s eta 0:00:44
   ---------------------------------------- 0.9/101.7 MB 2.5 MB/s eta 0:00:41
    --------------------------------------- 1.4/101.7 MB 3.2 MB/s eta 0:00:32
    ------------------

In [21]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Calculate imbalance ratio for class 1
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative:", negative_count)
print("Positive:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative: 13270
Positive: 154
scale_pos_weight: 86.16883116883118


In [22]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [23]:
y_pred_xgb = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.99      0.70      0.82      3318
           1       0.02      0.47      0.03        38

    accuracy                           0.70      3356
   macro avg       0.50      0.59      0.43      3356
weighted avg       0.98      0.70      0.81      3356



In [24]:
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred_xgb_threshold = (y_proba_xgb >= threshold).astype(int)

    print("Threshold:", threshold)
    print(classification_report(y_test, y_pred_xgb_threshold))
    print("-" * 50)

Threshold: 0.1
              precision    recall  f1-score   support

           0       0.99      0.09      0.16      3318
           1       0.01      0.95      0.02        38

    accuracy                           0.10      3356
   macro avg       0.50      0.52      0.09      3356
weighted avg       0.98      0.10      0.16      3356

--------------------------------------------------
Threshold: 0.2
              precision    recall  f1-score   support

           0       0.99      0.17      0.29      3318
           1       0.01      0.92      0.02        38

    accuracy                           0.18      3356
   macro avg       0.50      0.54      0.16      3356
weighted avg       0.98      0.18      0.28      3356

--------------------------------------------------
Threshold: 0.3
              precision    recall  f1-score   support

           0       1.00      0.27      0.43      3318
           1       0.01      0.92      0.03        38

    accuracy                       